In [ ]:
!nvidia-smi

Fri Oct  9 07:28:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# REQUIREMENTS
import os
os.environ["LLM_MODEL"] = "BAAI/BGE-VL-Screenshot"      # FORMAT IS NOT JUST THE MODEL NAME

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("vllm") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "vllm"])
else:
    print("vLLM is already installed.")

In [ ]:
import vllm
print(vllm.__version__)

0.31.0


In [ ]:
!pip uninstall -y torchaudio

Found existing installation: torchaudio 2.11.0+cu130
Uninstalling torchaudio-2.11.0+cu130:
  Successfully uninstalled torchaudio-2.11.0+cu130


In [ ]:

!vllm serve ${LLM_MODEL} \
    --runner pooling \
    --host 0.0.0.0 \
    --port 8000 \
    --gpu-memory-utilization 0.90 \
    --max-model-len 70000

(APIServer pid=6412) INFO 10-07 14:17:29 [api_utils.py:347] 
(APIServer pid=6412) INFO 10-07 14:17:29 [api_utils.py:347]        █     █     █▄   ▄█
(APIServer pid=6412) INFO 10-07 14:17:29 [api_utils.py:347]  ▄▄ ▄█ █     █     █ ▀▄▀ █  version 0.31.0
(APIServer pid=6412) INFO 10-07 14:17:29 [api_utils.py:347]   █▄█▀ █     █     █     █  model   BAAI/BGE-VL-Screenshot
(APIServer pid=6412) INFO 10-07 14:17:29 [api_utils.py:347]    ▀▀  ▀▀▀▀▀ ▀▀▀▀▀ ▀     ▀
(APIServer pid=6412) INFO 10-07 14:17:29 [api_utils.py:347] 
(APIServer pid=6412) INFO 10-07 14:17:29 [api_utils.py:286] non-default args: {'model_tag': 'BAAI/BGE-VL-Screenshot', 'host': '0.0.0.0', 'model': 'BAAI/BGE-VL-Screenshot', 'runner': 'pooling', 'max_model_len': 70000, 'gpu_memory_utilization': 0.9}
config.json: 100% 1.28k/1.28k [00:00<00:00, 2.21MB/s]
sentence_bert_config.json: 100% 748/748 [00:00<00:00, 853kB/s]
(APIServer pid=6412) INFO 10-07 14:17:30 [config.py:1131] Found sentence-transformers tokenize configuration.
(APISer

Stop the above cell once the "application startup complete" appears . This will stop a foreground running model engine.  Execute the next step to create a subprocess that runs the same engine but allows us to run other cells while our engine/ LLM is running.

In [ ]:
import subprocess

model_name = os.getenv('LLM_MODEL')

process = subprocess.Popen(
    [
        "vllm",
        "serve",
        model_name,
        "--runner", "pooling",
        "--host", "0.0.0.0",
        "--port", "8000",
        "--gpu-memory-utilization","0.90",
        "--max-model-len", "70000"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print("vLLM started. PID:", process.pid)

vLLM started. PID: 2240


In [ ]:

!curl http://localhost:8000/v1/models

curl: (7) Failed to connect to localhost port 8000 after 0 ms: Couldn't connect to server


In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

Selecting previously unselected package cloudflared.
(Reading database ... 127332 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.10.0) ...
Setting up cloudflared (2026.10.0) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [ ]:
# Add cloudflare gpg key
!sudo mkdir -p --mode=0755 /usr/share/keyrings
!curl -fsSL https://pkg.cloudflare.com/cloudflare-main.gpg | sudo tee /usr/share/keyrings/cloudflare-main.gpg >/dev/null

# Add this repo to your apt repositories
!echo 'deb [signed-by=/usr/share/keyrings/cloudflare-main.gpg] https://pkg.cloudflare.com/cloudflared any main' | sudo tee /etc/apt/sources.list.d/cloudflared.list

# install cloudflared
!sudo apt-get update && sudo apt-get install cloudflared

deb [signed-by=/usr/share/keyrings/cloudflare-main.gpg] https://pkg.cloudflare.com/cloudflared any main
Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease [1,578 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:4 https://pkg.cloudflare.com/cloudflared any InRelease [5,039 B]
Hit:5 http://archive.ubuntu.com/ubuntu noble InRelease
Get:6 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  Packages [1,912 kB]
Get:8 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:10 https://pkg.cloudflare.com/cloudflared any/main amd64 Packages [376 B]
Get:11 https://r2u.stat.illinois.edu/ubuntu noble/main amd64 Packages [3,052 kB]
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa

In [ ]:
import subprocess

TUNNEL_TOKEN = "eyJhIjoiNTNlYTU5ZGU4Y2RhMGRmM2RiZDg3ZGJlNGJlZTdmMGIiLCJ0IjoiN2ExNmZiMTgtMGQ5NC00MWZhLTkyOTYtMzFkNDg4MWZkN2ZiIiwicyI6IllXTTVNR1ZtT1RJdFltSm1NUzAwT0RWaExXSTRNbUV0WVRabE1XRmxZelV6WkRjNCJ9"

tunnel = subprocess.Popen(
    [
        "cloudflared",
        "tunnel",
        "run",
        "--token",
        TUNNEL_TOKEN
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# for line in tunnel.stdout:
#     print(line, end="")

In [ ]:
# import subprocess
# import re
# import time

# import re
# pattern = r"https?://[A-Za-z0-9.-]+\.trycloudflare\.com\b"
# tunnel = subprocess.Popen(
#     [
#         "cloudflared",
#         "tunnel",
#         "--url",
#         "http://localhost:8080"
#     ],
#     stdout=subprocess.PIPE,
#     stderr=subprocess.STDOUT,
#     text=True
# )
# link = ""
# for _ in range(30):
#     line = tunnel.stdout.readline()


#     if line:
#         print(line.strip())
#         matches = re.findall(pattern,line)
#         if matches:
#             link = matches[0]

# print("url link: " + link)
    # if "trycloudflare.com" in line:
    #     break

In [ ]:
import requests
req_link = requests.get('http://embedding.dripire.com/v1/models')

print(req_link.status_code)
print(req_link.text)

200
{"object":"list","data":[{"id":"BAAI/BGE-VL-Screenshot","object":"model","created":1791531764,"owned_by":"vllm","root":"BAAI/BGE-VL-Screenshot","parent":null,"max_model_len":70000,"permission":[{"id":"modelperm-a731becce0ca9e59","object":"model_permission","created":1791531764,"allow_create_engine":false,"allow_sampling":true,"allow_logprobs":true,"allow_search_indices":false,"allow_view":true,"allow_fine_tuning":false,"organization":"*","group":null,"is_blocking":false}]}]}


In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url= f"http://embedding.dripire.com/v1",
    api_key= "@759"
)

response = client.embeddings.create(
    model=model_name,
    input="Hello!"  # Embeddings accept 'input' (a string or list of strings) instead of 'messages'
)

embedding_vector = response.data[0].embedding
print(f"Generated a vector of length: {len(embedding_vector)}")
print(embedding_vector[:5])

# print(response.choices[0].message.content)

Generated a vector of length: 2048
[0.0062123560346663, -0.012223749421536922, -0.0084098344668746, -0.0008224163320846856, -0.004954157397150993]


In [ ]:
# !kill 16327
os.environ["LLM_BASE_URL"] = f"{link}/v1"
os.environ["LLM_API_KEY"] = "kkd390qoifoqoq2iof22j0dqd2dqgq"

In [ ]:
import os
LLM_BASE_URL = os.environ["LLM_BASE_URL"]
LLM_API_KEY = os.environ["LLM_API_KEY"]
LLM_MODEL = os.environ["LLM_MODEL"]

In [ ]:
# port killer
!lsof -i :8080


In [ ]:
!kill -9